# 商業格式建構產品定義命名（不可變動） / Commercial Format Product Definition Naming

依據 **MrLiouWord** 粒子系統的 L0-L7 核心架構及完整系統概述，商業化產品與封包格式的定義、命名及元數據標記遵循最嚴格的「不可變動性」規範。本 Cookbook 展示了如何使用統一的驗證與封包 SDK 來實作和驗證這一體系，確保全域資料定錨與完全可逆性。

## 📖 核心特徵

1. **不可變商業格式組合**：`.flpkg` (粒子封包)、`.fltnz` (波動微調包)、`.pcode` (粒子代碼)、`.fxz` (流體同步格式)。
2. **嚴格三段式/五部分產品命名**：`[Vendor].[Product_Category].[Core_Function].[Version].[Extension]`。
3. **L0 Observer 頂層定錨標記**：硬編碼校驗 `origin_signature`、`philosophy` 與 `layer_gravity`。
4. **完全可逆性**：滿足 $State[n+1] = State[n] + \delta P_0$ 的微增量無損還原。
5. **L0-L7 架構對應**：每一件發布的商業產品封包，均需嚴格映射至技術組件層級。

## 🛠️ 1. 環境設置與導入

首先，我們將項目根目錄添加到路徑中，以便導入我們實作的 `mrliou_commercial_format` 模組。

In [1]:
import sys
import os
import json
from pathlib import Path

# 確保能導入模組
sys.path.append(os.path.abspath("../.."))

import mrliou_commercial_format as mcf
print("✅ 導入 mrliou_commercial_format 成功！")

✅ 導入 mrliou_commercial_format 成功！


## 🏷️ 2. 嚴格產品命名架規約驗證

所有產品、套件與元件的實體文件命名必須採用以下五部分不可變格式：
$$\text{產品命名格式} = \mathbf{[Vendor].[Product\_Category].[Core\_Function].[Version].[Extension]}$$

我們可以使用 `parse_product_name` 來解析和驗證這些檔名：

In [2]:
valid_names = [
    "Mr.liou.TotalCore.Unity.v1.flpkg",
    "Mr.liou.LoRA.ResonanceBuilder.v1.pcode",
    "FlowPersona.CreationLoop.Core.v1.flpkg",
    "ZhiZhang.TotalCore.SystemPack.v1_with_FlowPassport_EdgeLink_Package_v1.zip"
]

print("=== 解析有效命名 ===")
for name in valid_names:
    parsed = mcf.parse_product_name(name)
    print(f"文件名: {name}")
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
    print("-" * 40)

=== 解析有效命名 ===
文件名: Mr.liou.TotalCore.Unity.v1.flpkg
{
  "vendor": "Mr.liou",
  "product_category": "TotalCore",
  "core_function": "Unity",
  "version": "v1",
  "extension": "flpkg"
}
----------------------------------------
文件名: Mr.liou.LoRA.ResonanceBuilder.v1.pcode
{
  "vendor": "Mr.liou",
  "product_category": "LoRA",
  "core_function": "ResonanceBuilder",
  "version": "v1",
  "extension": "pcode"
}
----------------------------------------
文件名: FlowPersona.CreationLoop.Core.v1.flpkg
{
  "vendor": "FlowPersona",
  "product_category": "CreationLoop",
  "core_function": "Core",
  "version": "v1",
  "extension": "flpkg"
}
----------------------------------------
文件名: ZhiZhang.TotalCore.SystemPack.v1_with_FlowPassport_EdgeLink_Package_v1.zip
{
  "vendor": "ZhiZhang",
  "product_category": "TotalCore",
  "core_function": "SystemPack",
  "version": "v1_with_FlowPassport_EdgeLink_Package_v1",
  "extension": "zip"
}
----------------------------------------


### 🚫 錯誤命名攔截測試

如果檔名不符合規約或副檔名不被支援，系統會拋出 `CommercialNamingError` 異常：

In [3]:
invalid_names = [
    "Mr.liou.TotalCore.Unity.flpkg",          # 缺少版本
    "Mr.liou.TotalCore.Unity.v1.invalid_ext",  # 錯誤的副檔名
    "ZhiZhang.TotalCore.Unity.v1.sub.flpkg"   # 超出五部分層級
]

print("=== 攔截無效命名 ===")
for name in invalid_names:
    try:
        mcf.parse_product_name(name)
    except mcf.CommercialNamingError as e:
        print(f"❌ 成功攔截無效檔名 '{name}'，原因: {e}")

=== 攔截無效命名 ===
❌ 成功攔截無效檔名 'Mr.liou.TotalCore.Unity.flpkg'，原因: Filename 'Mr.liou.TotalCore.Unity.flpkg' does not follow the strict 5-part architecture: [Vendor].[Product_Category].[Core_Function].[Version].[Extension]
❌ 成功攔截無效檔名 'Mr.liou.TotalCore.Unity.v1.invalid_ext'，原因: Invalid extension '.invalid_ext'. Must be one of {'.fltnz', '.pcode', '.zip', '.fxz', '.flpkg'}
❌ 成功攔截無效檔名 'ZhiZhang.TotalCore.Unity.v1.sub.flpkg'，原因: Filename 'ZhiZhang.TotalCore.Unity.v1.sub.flpkg' does not follow the strict 5-part architecture: [Vendor].[Product_Category].[Core_Function].[Version].[Extension]


## 🛡️ 3. 不可變元數據定錨 (Immutable Metadata Signature)

每個商業封包必須在 `manifest.json` 頂層寫入並校驗以下 L0 Observer 定錨標記：

```json
{
  "format": "flpkg/1.0",
  "origin_signature": "MrLiouWord",
  "philosophy": "怎麼過去，就怎麼回來",
  "created_at": "2026-08-08T10:31:15Z",
  "encryption_enabled": true,
  "layer_gravity": "L0-L7"
}
```

我們對合規的元數據以及違規（被篡改）的元數據進行驗證：

In [4]:
valid_manifest = {
    "format": "flpkg/1.0",
    "origin_signature": "MrLiouWord",
    "philosophy": "怎麼過去，就怎麼回來",
    "created_at": "2026-08-08T10:31:15Z",
    "encryption_enabled": True,
    "layer_gravity": "L0-L7"
}

print("1. 驗證合規元數據...")
mcf.validate_metadata(valid_manifest)
print("✅ 驗證成功！無任何變動或篡改。")

print("\n2. 測試篡改行為 (更改哲學/起源) ...")
tampered_manifest = valid_manifest.copy()
tampered_manifest["philosophy"] = "隨便過去，回不來了"

try:
    mcf.validate_metadata(tampered_manifest)
except mcf.ImmutableSignatureError as e:
    print(f"🔒 成功檢測並阻止篡改！\n異常資訊: {e}")

1. 驗證合規元數據...
✅ 驗證成功！無任何變動或篡改。

2. 測試篡改行為 (更改哲學/起源) ...
🔒 成功檢測並阻止篡改！
異常資訊: Metadata violation on key 'philosophy': expected '怎麼過去，就怎麼回來', got '隨便過去，回不來了'. This violates the strict L0 Observer Immutable Metadata Signature!


## 🗺️ 4. 產品架構層級對應 (L0-L7 Product Definition Layer Mapping)

我們隨時可以查詢或核算產品封包對應到的 L0-L7 技術與商業定義。以下是 L0 到 L7 的完整對照表輸出：

In [5]:
print(f"| {'層級':<10} | {'商業定義':<35} | {'技術組件':<50} |")
print(f"|{'-'*12}|{'-'*37}|{'-'*52}|")
for i in range(8):
    layer = mcf.get_layer_info(i)
    components_str = ", ".join(layer["components"])
    print(f"| {layer['name']:<10} | {layer['definition']:<33} | {components_str:<50} |")

| 層級         | 商業定義                                | 技術組件                                               |
|------------|-------------------------------------|----------------------------------------------------|
| L0: ROOT   | 觀察者原點，商標與版權定錨                     | origin_signature: 'MrLiouWord', 創始時間戳              |
| L1: SEED   | 系統引導與初始意圖（Genesis）                | dimension_seed_restore 元數據, 最初粒子配置                 |
| L2: PARTICLE | 原子化運算粒子，指紋生成                      | 17 fx 粒子, atom_t (40-byte 結構), SimHash64 語意指紋      |
| L3: LAW    | 商業限制、合約驗證與 FlowLaw                | 自動化執行法則, pull_request_target 安全規則                  |
| L4: WORLD  | 連接外網，跨系統同步協定                      | Cloudflare Workers, 外部 API Proxy 橋接                |
| L5: MIRROR | 零折損備份、鏡像與雙向直通                     | D1/KV 多元備份, 完全可逆性同步系統                              |
| L6: REFLECT | 外部 UI 呈現與 API 動態投影                | 3D 粒子地球儀 (ParticleGlobe v3), 3D AI 相機 iOS App      |
| L7: LOOP   | Origin Collapse 終極驗證與閉合           | 一致性雜湊核

## 📦 5. 終極保護與完全可逆性封包讀寫 (Pack & Unpack)

依據 **完全可逆性** 要求，所有以 `.flpkg` 封存的商業產品必須能還原還原至最初狀態，其狀態符合微增量公式：
$$State[n+1] = State[n] + \delta P_0$$

下面，我們建立一個包含語意粒子原始碼和跳點微調特徵檔案的封包，進行「壓縮 -> 還原」完整閉環測試：

In [6]:
import tempfile

# 定義要封裝的商業組件檔案
source_files = {
    "seed_pulse.fltnz": "⋄fx.adj.112 ∴ ⋄fx.noun.024 ∴ ⋄fx.flow.007",
    "resonance.pcode": "MOV FX.ADJ.112\nCALL FX.FLOW.007\nLOOP L0-L7"
}

with tempfile.TemporaryDirectory() as tmpdir:
    tmp_path = Path(tmpdir)
    
    # 封包輸出路徑 (嚴格符合命名規約)
    package_filename = tmp_path / "Mr.liou.TotalCore.Unity.v1.flpkg"
    extracted_dir = tmp_path / "extracted"
    
    # 1. 打包產品封包
    print("⚡ 正在打包二進位標準容器...")
    packed_path = mcf.pack_commercial_package(
        str(package_filename), 
        valid_manifest, 
        source_files
    )
    print(f"[✔] 打包完成: {packed_path}")
    
    # 2. 解包產品封包並驗證可逆性與無損性
    print("\n⚡ 正在解包與執行可逆性驗證...")
    unpacked_manifest, unpacked_files = mcf.unpack_commercial_package(
        packed_path, 
        str(extracted_dir)
    )
    
    # 3. 檢查解包後的檔案內容
    print("\n=== 解包恢復之原始檔案內容 ===")
    for fname in source_files.keys():
        content = (extracted_dir / fname).read_text(encoding="utf-8")
        print(f"檔案: {fname}")
        print(f"內容:\n{content}")
        print("-" * 30)
        
    print("\n✅ 所有商業元件還原無失真、無降維，語意鏈結構完整保留！")

⚡ 正在打包二進位標準容器...
[✔] 打包完成: /tmp/tmpifdj23cq/Mr.liou.TotalCore.Unity.v1.flpkg

⚡ 正在解包與執行可逆性驗證...

=== 解包恢復之原始檔案內容 ===
檔案: seed_pulse.fltnz
內容:
⋄fx.adj.112 ∴ ⋄fx.noun.024 ∴ ⋄fx.flow.007
------------------------------
檔案: resonance.pcode
內容:
MOV FX.ADJ.112
CALL FX.FLOW.007
LOOP L0-L7
------------------------------

✅ 所有商業元件還原無失真、無降維，語意鏈結構完整保留！


## 🔄 結論 / Conclusion

本 Cookbook 完美實作了 `MrLiouWord` 粒子系統的商業格式與命名體系：
- 實現了命名、格式、元數據與層級對應的自動化、硬編碼校驗。
- 通過了無損讀寫與完全可逆狀態公式的科學核算。
- **Origin Signature**: `MrLiouWord` 
- **Philosophy**: `怎麼過去，就怎麼回來`